<a href="https://colab.research.google.com/github/JONNY-ME/multilingual-health-qa/blob/main/03_gemma4_31b_unsloth_qlora_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 31B Unsloth QLoRA Fine-Tuning

This notebook adapts the Unsloth Gemma 4 fine-tuning workflow to the multilingual health QA task.

It trains `unsloth/gemma-4-31B-it` with LoRA/QLoRA on curated train question-answer pairs, validates with ROUGE-1 and ROUGE-L, and creates prediction files with the required target columns.

Recommended runtime: Colab Pro+ with A100 or another high-memory GPU.

## 1. Runtime Setup

Run on a GPU runtime. The source Unsloth notebook uses the Unsloth stack for Gemma 4, including `FastModel`, LoRA adapters, and TRL SFT training.

If the runtime already imported `torch`, `transformers`, or `unsloth` before this install cell, restart the runtime after installation and run from the top.

In [1]:
# Check GPU. This should show an NVIDIA GPU in Colab.
!nvidia-smi

# Deterministic clean install for Unsloth Gemma 4.
# This avoids the CUDA 13 / bitsandbytes libnvJitLink conflict by pinning the PyTorch CUDA 12.8 stack.
# IMPORTANT: Run this cell, then restart the runtime, then run the notebook from the top.
!pip -q install uv

!uv pip uninstall --system -q -y torch torchvision torchaudio triton bitsandbytes transformers datasets trl peft accelerate unsloth unsloth_zoo xformers flash-attn || true

# Install PyTorch CUDA 12.8 wheels first. CUDA 12.8 is compatible with Blackwell and avoids the CUDA 13 bnb ABI break.
!uv pip install --system -q --index-url https://download.pytorch.org/whl/cu128 torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0

# Install the rest from PyPI with pinned versions known to work together for Gemma 4 + Unsloth.
!uv pip install --system -q --upgrade triton==3.6.0 transformers==5.5.0 datasets==4.3.0 trl==0.24.0 peft==0.19.1 accelerate==1.13.0 unsloth==2026.5.4 unsloth_zoo==2026.5.2 pandas numpy tqdm rouge-score sentencepiece huggingface_hub gdown safetensors torchcodec timm protobuf

import torch, transformers
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers version:", transformers.__version__)
print("Setup complete. Restart the runtime now, then run from the top.")

Tue May 19 06:51:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
# Runtime sanity check. Do this before importing Unsloth.
# This notebook intentionally avoids 4-bit loading and 8-bit optimizer by default,
# so bitsandbytes is not required for the main training path.
import os
import torch

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

# Hard fail early if the runtime was not restarted after dependency installation.
try:
    import transformers
    print("Transformers version:", transformers.__version__)
except Exception as exc:
    raise RuntimeError("Transformers import failed. Restart runtime after setup and run from the top.") from exc

Torch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True
Transformers version: 5.5.0


In [1]:
import gc
import json
import os
import random
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 240)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Data Download and Loading

The notebook downloads the CSV files into `/content/multilingual-health-question-answering/data` when running in Colab. If the files already exist, the download step is skipped.

In [2]:
import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1gqWeNhgZGqqJtemvn9a1pMuEjVmi2s0t?usp=sharing"
COLAB_PROJECT_DIR = Path("/content/multilingual-health-question-answering")
COLAB_DATA_DIR = COLAB_PROJECT_DIR / "data"
EXPECTED_DATA_FILES = ["Train.csv", "Val.csv", "Test.csv", "SampleSubmission.csv"]

COLAB_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_before = [name for name in EXPECTED_DATA_FILES if not (COLAB_DATA_DIR / name).exists()]
if missing_before:
    print("Downloading missing data files:", missing_before)
    gdown.download_folder(
        url=DRIVE_FOLDER_URL,
        output=str(COLAB_DATA_DIR),
        quiet=False,
        use_cookies=False,
    )
else:
    print("All expected data files already exist. Skipping download.")

# gdown may preserve a nested Drive folder name. Normalize files into COLAB_DATA_DIR.
for filename in EXPECTED_DATA_FILES:
    target = COLAB_DATA_DIR / filename
    if target.exists():
        continue
    matches = [path for path in COLAB_PROJECT_DIR.rglob(filename) if path.is_file()]
    if matches:
        shutil.copy2(matches[0], target)

missing_after = [name for name in EXPECTED_DATA_FILES if not (COLAB_DATA_DIR / name).exists()]
if missing_after:
    raise FileNotFoundError(f"Could not find expected data files after download: {missing_after}")

for filename in EXPECTED_DATA_FILES:
    path = COLAB_DATA_DIR / filename
    print(f"- {path} ({path.stat().st_size / 1024**2:.2f} MB)")

All expected data files already exist. Skipping download.
- /content/multilingual-health-question-answering/data/Train.csv (18.27 MB)
- /content/multilingual-health-question-answering/data/Val.csv (4.26 MB)
- /content/multilingual-health-question-answering/data/Test.csv (0.34 MB)
- /content/multilingual-health-question-answering/data/SampleSubmission.csv (0.24 MB)


In [3]:
PROJECT_DIR_CANDIDATES = [
    Path("/content/multilingual-health-question-answering"),
    Path("/content/drive/MyDrive/multilingual-health-question-answering"),
    Path.cwd(),
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if (candidate / "data" / "Train.csv").exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find data/Train.csv. Set PROJECT_DIR manually.")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Train.csv"
VAL_PATH = DATA_DIR / "Val.csv"
TEST_PATH = DATA_DIR / "Test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub_df = pd.read_csv(SAMPLE_SUB_PATH)

REQUIRED_COLUMNS = {
    "train": {"ID", "input", "output", "subset"},
    "val": {"ID", "input", "output", "subset"},
    "test": {"ID", "input", "subset"},
    "sample_submission": {"ID", "TargetRLF1", "TargetR1F1", "TargetLLM"},
}

for name, df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
    "sample_submission": sample_sub_df,
}.items():
    missing = REQUIRED_COLUMNS[name] - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

assert test_df["ID"].tolist() == sample_sub_df["ID"].tolist(), "Test IDs do not match sample submission order."

print("PROJECT_DIR:", PROJECT_DIR)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
display(train_df["subset"].value_counts().sort_index().to_frame("train_rows"))

PROJECT_DIR: /content/multilingual-health-question-answering
Train: (29815, 4)
Validation: (6686, 4)
Test: (2618, 3)


,train_rows
subset,
Aka_Gha,4455
Amh_Eth,1845
Eng_Eth,3915
Eng_Gha,4443
Eng_Ken,2080
Eng_Uga,7624
Lug_Uga,3383
Swa_Ken,2070


## 3. Conversation Formatting

Each training row becomes a two-turn chat:

- user: subset/language cue plus the health question
- assistant: the curated reference answer

The training loss is later masked so the model learns from assistant answers only.

In [4]:
SUBSET_HINTS = {
    "Aka_Gha": "Akan/Twi as used in Ghana",
    "Lug_Uga": "Luganda as used in Uganda",
    "Swa_Ken": "Kiswahili as used in Kenya",
    "Amh_Eth": "Amharic as used in Ethiopia",
    "Eng_Uga": "English as used in Uganda",
    "Eng_Gha": "English as used in Ghana",
    "Eng_Ken": "English as used in Kenya",
    "Eng_Eth": "English as used in Ethiopia",
}

SYSTEM_PROMPT = """You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.
Answer accurately, clearly, and respectfully.
Do not invent medical facts, drug dosages, or legal requirements.
If the question is in a non-English African language, answer in that same language.
Return only the final answer text. Do not add labels, markdown, explanations, translations, or citations."""


def build_user_prompt(question: str, subset: str) -> str:
    language_hint = SUBSET_HINTS.get(subset, subset)
    return f"""Subset/language cue: {subset} ({language_hint})

Question:
{question}

Write a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer."""


def row_to_conversation(row: pd.Series) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row["input"], row["subset"])},
        {"role": "assistant", "content": str(row["output"]).strip()},
    ]


def row_to_inference_messages(row: pd.Series) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row["input"], row["subset"])},
    ]

example_conversation = row_to_conversation(train_df.iloc[0])
example_conversation

[{'role': 'system',
  'content': 'You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.\nAnswer accurately, clearly, and respectfully.\nDo not invent medical facts, drug dosages, or legal requirements.\nIf the question is in a non-English African language, answer in that same language.\nReturn only the final answer text. Do not add labels, markdown, explanations, translations, or citations.'},
 {'role': 'user',
  'content': 'Subset/language cue: Aka_Gha (Akan/Twi as used in Ghana)\n\nQuestion:\nƆkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn werɛ, sɛnea wɔbɛboa wɔn ma wɔanya mmoa firi nnwumakuo a ɛfata hɔ, ne sɛnea wɔbɛsiw afɔbu suban ne nsɛm a nkurɔfoɔ de gu oyarefoɔ no so no kwan.\n\nWrite a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer.'},

In [5]:
# Start with a sample for a fast proof run. Set TRAIN_SAMPLE_SIZE = None for all training rows.
TRAIN_SAMPLE_SIZE = 3000

train_work_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
if TRAIN_SAMPLE_SIZE is not None:
    train_work_df = train_work_df.head(TRAIN_SAMPLE_SIZE).copy()

train_work_df["conversations"] = train_work_df.apply(row_to_conversation, axis=1)
train_dataset = Dataset.from_pandas(train_work_df[["ID", "subset", "conversations"]], preserve_index=False)

print("Training rows:", len(train_dataset))
print(train_dataset[0]["conversations"])

Training rows: 3000
[{'role': 'system', 'content': 'You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.\nAnswer accurately, clearly, and respectfully.\nDo not invent medical facts, drug dosages, or legal requirements.\nIf the question is in a non-English African language, answer in that same language.\nReturn only the final answer text. Do not add labels, markdown, explanations, translations, or citations.'}, {'role': 'user', 'content': 'Subset/language cue: Amh_Eth (Amharic as used in Ethiopia)\n\nQuestion:\nቤተሰቤ ገንዘብ ከሌለው እንዴት በነፃ ወይም በዝቅተኛ ዋጋ ምርመራ ማድረግ እችላለሁ?\n\nWrite a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer.'}, {'role': 'assistant', 'content': 'በአቅራቢያዎ የሚገኘውን ክሊኒክ ነፃ የምርመራ ቀናት ካሏቸው ጠይቂ፣ ወይም በመንግሥት ወይም በበጎ አድራጎት ድርጅቶች የሚደገፉ የሕዝብ ጤና ፕሮግራሞችን ፈልጊ።'}]


## 4. Load Gemma 4 31B With Unsloth

This section follows the source Unsloth Gemma 4 notebook but keeps only text fine-tuning. Vision layers are disabled for LoRA because this task is text-only.

In [6]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

MODEL_ID = "unsloth/gemma-4-31B-it"
FALLBACK_MODEL_ID = "unsloth/gemma-4-26B-A4B-it"
MAX_SEQ_LENGTH = 2048  # Answers are short; lower than 8192 to reduce memory during SFT.
LOAD_IN_4BIT = False  # Avoid bitsandbytes dependency conflicts; 31B BF16 fits on the 96GB GPU runtime.
FULL_FINETUNING = False

print("MODEL_ID:", MODEL_ID)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("LOAD_IN_4BIT:", LOAD_IN_4BIT)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
MODEL_ID: unsloth/gemma-4-31B-it
MAX_SEQ_LENGTH: 2048
LOAD_IN_4BIT: False


In [7]:
def clear_gpu_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


clear_gpu_memory()

try:
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        dtype=None,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        full_finetuning=FULL_FINETUNING,
    )
except Exception as exc:
    print(f"Failed to load {MODEL_ID}: {exc}")
    print(f"Falling back to {FALLBACK_MODEL_ID}")
    MODEL_ID = FALLBACK_MODEL_ID
    clear_gpu_memory()
    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        dtype=None,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        full_finetuning=FULL_FINETUNING,
    )

# Gemma 4 chat template support comes from Unsloth.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4-thinking",
)

print("Loaded model:", MODEL_ID)
if torch.cuda.is_available():
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))

==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Loaded model: unsloth/gemma-4-31B-it
Allocated GB: 58.27
Reserved GB: 58.27


In [8]:
LORA_R = 8
LORA_ALPHA = 8
LORA_DROPOUT = 0

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    random_state=SEED,
)

print("LoRA adapters attached.")

LoRA adapters attached.


In [9]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=["conversations"],
)

print(train_dataset[0]["text"][:2000])

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

<|turn>system
You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.
Answer accurately, clearly, and respectfully.
Do not invent medical facts, drug dosages, or legal requirements.
If the question is in a non-English African language, answer in that same language.
Return only the final answer text. Do not add labels, markdown, explanations, translations, or citations.<turn|>
<|turn>user
Subset/language cue: Amh_Eth (Amharic as used in Ethiopia)

Question:
ቤተሰቤ ገንዘብ ከሌለው እንዴት በነፃ ወይም በዝቅተኛ ዋጋ ምርመራ ማድረግ እችላለሁ?

Write a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer.<turn|>
<|turn>model
በአቅራቢያዎ የሚገኘውን ክሊኒክ ነፃ የምርመራ ቀናት ካሏቸው ጠይቂ፣ ወይም በመንግሥት ወይም በበጎ አድራጎት ድርጅቶች የሚደገፉ የሕዝብ ጤና ፕሮግራሞችን ፈልጊ።<turn|>



## 5. Train LoRA Adapter

The default settings are for a proof run. Increase `MAX_STEPS` or set `NUM_TRAIN_EPOCHS` for a longer run after the validation sample improves over the zero-shot baseline.

In [10]:
from trl import SFTConfig, SFTTrainer

RUN_TRAINING = True
MAX_STEPS = 1000
NUM_TRAIN_EPOCHS = None  # Set to 1 and MAX_STEPS = -1 for a full epoch.
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
OPTIMIZER = "adamw_torch"  # Avoid adamw_8bit and keep bitsandbytes out of the training path.

ADAPTER_DIR = OUTPUT_DIR / "gemma4_31b_unsloth_lora"
CHECKPOINT_DIR = OUTPUT_DIR / "gemma4_31b_unsloth_checkpoints"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

sft_args = SFTConfig(
    dataset_text_field="text",
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=5,
    max_steps=MAX_STEPS if NUM_TRAIN_EPOCHS is None else -1,
    num_train_epochs=NUM_TRAIN_EPOCHS if NUM_TRAIN_EPOCHS is not None else 1,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    optim=OPTIMIZER,
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=SEED,
    output_dir=str(CHECKPOINT_DIR),
    save_strategy="steps",
    save_steps=50,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,
    args=sft_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

print("Trainer ready.")
print("Train rows:", len(train_dataset))
print("Max steps:", MAX_STEPS)
print("Optimizer:", OPTIMIZER)

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/3000 [00:00<?, ? examples/s]

Map (num_proc=52):   0%|          | 0/3000 [00:00<?, ? examples/s]

Filter (num_proc=52):   0%|          | 0/3000 [00:00<?, ? examples/s]

Trainer ready.
Train rows: 3000
Max steps: 1000
Optimizer: adamw_torch


In [11]:
if RUN_TRAINING:
    start_time = time.time()
    start_gpu_memory = torch.cuda.max_memory_reserved() / 1024**3 if torch.cuda.is_available() else 0
    trainer_stats = trainer.train()
    elapsed = time.time() - start_time
    print(f"Training runtime: {elapsed / 60:.2f} minutes")
    if torch.cuda.is_available():
        used_memory = torch.cuda.max_memory_reserved() / 1024**3
        print(f"Peak reserved memory: {used_memory:.3f} GB")
        print(f"Training memory delta: {used_memory - start_gpu_memory:.3f} GB")
else:
    trainer_stats = None
    print("RUN_TRAINING is False. Skipping training.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 2 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 61,214,720 of 31,334,301,232 (0.20% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.908900
2,1.375224
3,0.754489
4,1.032419
5,0.998814
6,0.866703
7,1.022624
8,0.850234
9,0.884276
10,0.502420


Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_checkpoints/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_uns

Training runtime: 24.97 minutes
Peak reserved memory: 65.002 GB
Training memory delta: 6.498 GB


In [12]:
# Save only the LoRA adapter. This is small enough to keep as a run artifact.
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved adapter to:", ADAPTER_DIR)

Unsloth: Restored added_tokens_decoder metadata in /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_lora/tokenizer_config.json.


Saved adapter to: /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_lora


## 6. Inference Utilities

The generation path mirrors the zero-shot notebooks, but uses the fine-tuned Unsloth model and tokenizer.

In [13]:
THINKING_PATTERNS = [
    r"<\|channel\>thought.*?(?=<\|channel\>|$)",
    r"<think>.*?</think>",
    r"<\|think\|>",
]
SPECIAL_TOKEN_PATTERN = re.compile(r"<\|[^>]+\|>|</?s>|<pad>|<bos>|<eos>")
ROLE_PREFIX_PATTERN = re.compile(
    r"^(assistant|model|answer|final answer|response)\s*[:\-]\s*",
    flags=re.IGNORECASE,
)


def clean_generation(text: str) -> str:
    if text is None:
        return ""
    cleaned = str(text)
    for pattern in THINKING_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.DOTALL | re.IGNORECASE)
    cleaned = SPECIAL_TOKEN_PATTERN.sub(" ", cleaned)
    cleaned = cleaned.replace("```", " ")
    cleaned = ROLE_PREFIX_PATTERN.sub("", cleaned.strip()).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip(' \"\'')


@torch.inference_mode()
def generate_answer(
    row: pd.Series,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    repetition_penalty: float = 1.05,
) -> str:
    messages = row_to_inference_messages(row)
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    ).to(model.device)
    input_len = inputs.shape[-1]

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature and temperature > 0:
        generation_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": top_p})
    else:
        generation_kwargs["do_sample"] = False

    outputs = model.generate(input_ids=inputs, **generation_kwargs)
    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=False)
    return clean_generation(raw)

print("Inference utilities ready.")

Inference utilities ready.


In [14]:
def run_generation_with_checkpoints(
    df: pd.DataFrame,
    output_path: Path,
    checkpoint_every: int = 25,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    overwrite: bool = False,
) -> pd.DataFrame:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        existing = pd.read_csv(output_path)
        pred_map = dict(zip(existing["ID"], existing["prediction"].fillna(""))) if {"ID", "prediction"}.issubset(existing.columns) else {}
    else:
        pred_map = {}

    rows = []
    start_time = time.time()
    for _, row in tqdm(df.iterrows(), total=len(df)):
        row_id = row["ID"]
        if row_id in pred_map and str(pred_map[row_id]).strip():
            prediction = pred_map[row_id]
        else:
            prediction = generate_answer(
                row,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
            pred_map[row_id] = prediction

        rows.append({
            "ID": row_id,
            "subset": row["subset"],
            "input": row["input"],
            "prediction": prediction,
        })

        if len(rows) % checkpoint_every == 0:
            pd.DataFrame(rows).to_csv(output_path, index=False)

    result_df = pd.DataFrame(rows)
    result_df.to_csv(output_path, index=False)
    print(f"Saved {len(result_df)} predictions to {output_path}")
    print(f"Elapsed: {(time.time() - start_time) / 60:.1f} minutes")
    return result_df

print("Checkpointed inference ready.")

Checkpointed inference ready.


## 7. Validation Scoring

Local validation reports ROUGE-1 F1 and ROUGE-L F1. The LLM judge score is not estimated here; compare public leaderboard scores separately in the run log.

In [15]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)


def score_pair(reference: str, prediction: str) -> dict[str, float]:
    reference = "" if pd.isna(reference) else str(reference)
    prediction = "" if pd.isna(prediction) else str(prediction)
    scores = rouge.score(reference, prediction)
    return {
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rougeL_f1": scores["rougeL"].fmeasure,
    }


def score_predictions(pred_df: pd.DataFrame, reference_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    merged = pred_df.merge(
        reference_df[["ID", "output"]],
        on="ID",
        how="left",
        validate="one_to_one",
    )
    score_rows = [score_pair(ref, pred) for ref, pred in zip(merged["output"], merged["prediction"])]
    scored = pd.concat([merged.reset_index(drop=True), pd.DataFrame(score_rows)], axis=1)
    scored["local_rouge_weighted"] = 0.37 * scored["rouge1_f1"] + 0.37 * scored["rougeL_f1"]

    summary = scored.groupby("subset").agg(
        rows=("ID", "count"),
        rouge1_f1=("rouge1_f1", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        local_rouge_weighted=("local_rouge_weighted", "mean"),
        pred_words=("prediction", lambda s: np.mean([len(str(x).split()) for x in s])),
        ref_words=("output", lambda s: np.mean([len(str(x).split()) for x in s])),
    ).reset_index()

    overall = pd.DataFrame([{
        "subset": "OVERALL",
        "rows": len(scored),
        "rouge1_f1": scored["rouge1_f1"].mean(),
        "rougeL_f1": scored["rougeL_f1"].mean(),
        "local_rouge_weighted": scored["local_rouge_weighted"].mean(),
        "pred_words": np.mean([len(str(x).split()) for x in scored["prediction"]]),
        "ref_words": np.mean([len(str(x).split()) for x in scored["output"]]),
    }])
    return scored, pd.concat([overall, summary], ignore_index=True)

print("Scoring utilities ready.")

Scoring utilities ready.


In [17]:
from typing import Any
import re
import pandas as pd
import torch

THINKING_PATTERNS = [
    r"<\|channel\>thought.*?(?=<\|channel\>|$)",
    r"<think>.*?</think>",
    r"<\|think\|>",
]
SPECIAL_TOKEN_PATTERN = re.compile(r"<\|[^>]+\|>|</?s>|<pad>|<bos>|<eos>")
ROLE_PREFIX_PATTERN = re.compile(
    r"^(assistant|model|answer|final answer|response)\s*[:\-]\s*",
    flags=re.IGNORECASE,
)

def clean_generation(text: str) -> str:
    if text is None:
        return ""
    cleaned = str(text)
    for pattern in THINKING_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.DOTALL | re.IGNORECASE)
    cleaned = SPECIAL_TOKEN_PATTERN.sub(" ", cleaned)
    cleaned = cleaned.replace("```", " ")
    cleaned = ROLE_PREFIX_PATTERN.sub("", cleaned.strip()).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip(' "\'')

# Redefine row_to_inference_messages to use the multimodal content format
def row_to_inference_messages(row: pd.Series) -> list[dict[str, Any]]:
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": build_user_prompt(row["input"], row["subset"])}]}
    ]

@torch.inference_mode()
def generate_answer(
    row: pd.Series,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    repetition_penalty: float = 1.05,
) -> str:
    messages = row_to_inference_messages(row)

    # Step 1: Get the raw string output from apply_chat_template
    raw_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False, # Do not tokenize here, just get the string
    )

    # Ensure raw_text is an empty string if apply_chat_template returns None
    raw_text = raw_text if raw_text is not None else ""

    # Step 2: Tokenize the raw string to get input_ids and attention_mask
    tokenized_output = tokenizer(
        text=raw_text, # Explicitly pass raw_text to the 'text' argument
        return_tensors="pt",
        return_attention_mask=True,
    ).to(model.device)

    inputs = tokenized_output["input_ids"]
    attention_mask = tokenized_output["attention_mask"]
    input_len = inputs.shape[-1]

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if temperature and temperature > 0:
        generation_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": top_p})
    else:
        generation_kwargs["do_sample"] = False

    # Pass both input_ids and attention_mask to model.generate
    outputs = model.generate(input_ids=inputs, attention_mask=attention_mask, **generation_kwargs)
    raw = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=False)
    return clean_generation(raw)

# Smoke validation: one row per subset.
smoke_df = (
    val_df.sample(frac=1.0, random_state=SEED)
    .groupby("subset", group_keys=False)
    .head(1)
    .sort_values("subset")
    .reset_index(drop=True)
)

SMOKE_PATH = OUTPUT_DIR / "gemma4_31b_unsloth_smoke_predictions.csv"
smoke_pred_df = run_generation_with_checkpoints(
    smoke_df,
    SMOKE_PATH,
    checkpoint_every=1,
    overwrite=True,
)
smoke_scored_df, smoke_summary_df = score_predictions(smoke_pred_df, smoke_df)
display(smoke_summary_df.round(4))
display(smoke_scored_df[["ID", "subset", "input", "output", "prediction", "rouge1_f1", "rougeL_f1"]])

  0%|          | 0/8 [00:00<?, ?it/s]

Saved 8 predictions to /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_smoke_predictions.csv
Elapsed: 1.1 minutes


,subset,rows,rouge1_f1,rougeL_f1,local_rouge_weighted,pred_words,ref_words
0,OVERALL,8,0.3641,0.2708,0.2349,76.125,97.375
1,Aka_Gha,1,0.5178,0.3350,0.3155,79.000,76.000
2,Amh_Eth,1,0.0000,0.0000,0.0000,17.000,17.000
3,Eng_Eth,1,0.4058,0.2899,0.2574,38.000,30.000
4,Eng_Gha,1,0.3056,0.2685,0.2124,165.000,49.000
5,Eng_Ken,1,0.5165,0.3846,0.3334,64.000,107.000
6,Eng_Uga,1,0.5458,0.4198,0.3573,171.000,337.000
7,Lug_Uga,1,0.3434,0.2828,0.2317,34.000,51.000
8,Swa_Ken,1,0.2781,0.1854,0.1715,41.000,112.000


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1
0,ID_VL_Aka_Gha_EB2263DD,Aka_Gha,Ɔkwan bɛn so na mmabun betumi akamfo wɔn ho ahwɛ ahu sɛ wobenya ayaresa a obu ne animhwɛ nnim afi akwaahosan ho adwumayɛfo hɔ?,"Ɛsɛ sɛ mmabun di kan kyerɛkyerɛ wɔn ho akwannya ahorow ho asɛm, na wɔkyerɛ hokwan a wɔwɔ sɛ wɔde obu ne animhwɛ nnim di dwuma denam wɔn ahiade ne nea wɔpɛ a wɔbɛka akyerɛ akwahosan ho adwumayɛfo pefee no so. Wobetumi ahwehwɛ akwahosan h...",Mmabun betumi akamfo wɔn ho ahwɛ ahu sɛ wobenya ayaresa a obu ne animhwɛ nnim denam: Nsɛm a wɔbɛka pefee ne nkitahodi a ɛyɛ nokware a wɔbɛka ho. Nhyehyɛe a wɔde wɔn ho bɛhyɛ mu de ahwehwɛ sɛ wɔbɛhwehwɛ mmoa a ɛfata na ɛyɛ kokoam. Nhyehy...,0.517766,0.335025
1,ID_VL_Amh_Eth_A5CB63A2,Amh_Eth,በሰውነቴ እና በመልክዬ ኩራት ይሰማኛል?,አይከንቱ ወይም ጉራ መሆን ስህተት ነው፣ ነገር ግን የሰውነትዎን ጥንካሬ እና ጤንነት ማድነቅ ለእግዚአብሔር ፍጥረት አክብሮት ያሳያል።,አዎ፣ በራስዎ መተማመን እና ኩራት ሊኖርዎት ይገባል። እያንዳንዱ ሰው ልዩ እና ውብ ነው። ራስዎን ከሌሎች ጋር አያወዳድሩ።<turn|>,0.000000,0.000000
2,ID_VL_Eng_Eth_01902620,Eng_Eth,"If I get the medicine for Gonorrhea, should my partner also take the medicine, even if he says he feels fine?","This is a question about, Partner Treatment. Yes, always. He may be asymptomatic (no signs), but he will still reinfect you. Both partners must be treated at the same time.","Yes. Both partners must be treated at the same time, even if one has no symptoms. If the partner is not treated, they can pass the infection back to you, and the disease can cause serious health problems.<turn|>",0.405797,0.289855
3,ID_VL_Eng_Gha_2CD2A404,Eng_Gha,"Are there specific healthcare providers or clinics that specialize in providing medical assistance to adolescents in crisis situations, and how can adolescents find them?","There are specific healthcare providers or clinics that specialize in providing medical assistance to adolescents in crisis situations, such as sexual health clinics, family planning clinics, or adolescent health centers. Adolescents ca...","Yes, there are healthcare providers and clinics that specialize in providing medical assistance to adolescents in crisis situations. These may include: Adolescent Health Centers: Many communities have specialized clinics or centers that...",0.305556,0.268519
4,ID_VL_Eng_Ken_0B531932,Eng_Ken,Can I have a child with a HIV patient without getting infected?,"Yes, it is possible for an HIV-negative individual to have a child with an HIV-positive partner without acquiring HIV if certain precautions and medical interventions are followed. The process typically involves a combination of strateg...","Yes, it is possible to have a child with an HIV-positive partner without getting infected with HIV. This can be achieved through various methods and precautions to prevent the transmission of HIV from the HIV-positive partner to the HIV...",0.516484,0.384615
5,ID_VL_Eng_Uga_629EF3D8,Eng_Uga,"How can I protect myself from contracting Herpes Simplex Virus?, please answer in detail.","The only way to completely avoid STIs is to not have vaginal, anal, or oral sex. If you are sexually active, you can do the following things to lower your chances of getting Genital Herpes or Herpes Simplex Virus (HSV), also known as he...","The only way to completely avoid STIs is to not have vaginal, anal, or oral sex. If you are sexually active, the following things can lower your chances of getting Genital Herpes or Herpes Simplex Virus (HSV), also known as herpes:• Bei...",0.545802,0.419847
6,ID_VL_Lug_Uga_AD409244,Lug_Uga,Nsobola ntya okuwangaala n'Akawuka akaleeta Siriimu/Siriimu?,"Okuwangaala n'okuweeweeza Akawuka akaleeta Siriimu kisobokera ddala n'obujjanjabi obutuufu, obujjanjabi n'engeri gy'osalawo okweyisa. Abantu abalina Akawuka akaleeta Siriimu basobola okuwangaala nako mu bulamu obweyagaza nga balina obuj...","Okubeera n'Akawuka akaleeta Siriimu tekitegeeza nti obulamu bwo bwe bukyuse. Abantu bangi abalina Akawuka akaleeta Siriimu basobola okuwangaala obulamu obulungi n'obw'ekitiibwa. Kino kiyinza okuba okukosa okukola ku

```
subset	rows	rouge1_f1	rougeL_f1	local_rouge_weighted	pred_words	ref_words
0	OVERALL	8	0.3090	0.1830	0.1821	102.875	97.375
1	Aka_Gha	1	0.5472	0.2547	0.2967	98.000	76.000
2	Amh_Eth	1	0.0000	0.0000	0.0000	25.000	17.000
3	Eng_Eth	1	0.3333	0.2444	0.2138	59.000	30.000
4	Eng_Gha	1	0.4072	0.3114	0.2659	119.000	49.000
5	Eng_Ken	1	0.4275	0.2609	0.2547	158.000	107.000
6	Eng_Uga	1	0.3391	0.1387	0.1768	171.000	337.000
7	Lug_Uga	1	0.0843	0.0723	0.0580	105.000	51.000
8	Swa_Ken	1	0.3333	0.1818	0.1906	88.000	112.000

```

In [18]:
# Stratified validation sample. Increase SAMPLE_PER_SUBSET when smoke output looks healthy.
SAMPLE_PER_SUBSET = 10
sample_val_df = (
    val_df.sample(frac=1.0, random_state=SEED)
    .groupby("subset", group_keys=False)
    .head(SAMPLE_PER_SUBSET)
    .sort_values(["subset", "ID"])
    .reset_index(drop=True)
)

SAMPLE_VAL_PATH = OUTPUT_DIR / f"gemma4_31b_unsloth_val_sample_{SAMPLE_PER_SUBSET}_per_subset.csv"
sample_pred_df = run_generation_with_checkpoints(
    sample_val_df,
    SAMPLE_VAL_PATH,
    checkpoint_every=10,
    overwrite=False,
)
sample_scored_df, sample_summary_df = score_predictions(sample_pred_df, sample_val_df)
display(sample_summary_df.round(4))

  0%|          | 0/80 [00:00<?, ?it/s]

Saved 80 predictions to /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_val_sample_10_per_subset.csv
Elapsed: 7.6 minutes


,subset,rows,rouge1_f1,rougeL_f1,local_rouge_weighted,pred_words,ref_words
0,OVERALL,80,0.3285,0.2429,0.2114,51.275,80.175
1,Aka_Gha,10,0.3914,0.2513,0.2378,62.000,97.500
2,Amh_Eth,10,0.0667,0.0667,0.0493,17.500,15.800
3,Eng_Eth,10,0.4186,0.3065,0.2683,23.300,23.600
4,Eng_Gha,10,0.3986,0.2747,0.2491,80.800,84.500
5,Eng_Ken,10,0.3515,0.2523,0.2234,59.800,91.500
6,Eng_Uga,10,0.4767,0.4008,0.3247,89.900,146.400
7,Lug_Uga,10,0.2015,0.1672,0.1364,29.600,78.700
8,Swa_Ken,10,0.3229,0.2238,0.2023,47.300,103.400


```

subset	rows	rouge1_f1	rougeL_f1	local_rouge_weighted	pred_words	ref_words
0	OVERALL	80	0.2973	0.2054	0.1860	76.2125	80.175
1	Aka_Gha	10	0.3543	0.2271	0.2151	88.4000	97.500
2	Amh_Eth	10	0.1800	0.1800	0.1332	33.2000	15.800
3	Eng_Eth	10	0.2477	0.1701	0.1546	71.5000	23.600
4	Eng_Gha	10	0.4283	0.2882	0.2651	108.6000	84.500
5	Eng_Ken	10	0.3301	0.2172	0.2025	76.9000	91.500
6	Eng_Uga	10	0.3394	0.2083	0.2027	103.4000	146.400
7	Lug_Uga	10	0.1653	0.1249	0.1074	59.7000	78.700
8	Swa_Ken	10	0.3336	0.2274	0.2076	68.0000	103.400

```

In [19]:
# Full validation is optional and can be slow.
RUN_FULL_VALIDATION = False

FULL_VAL_PATH = OUTPUT_DIR / "gemma4_31b_unsloth_val_predictions.csv"
FULL_VAL_SCORED_PATH = OUTPUT_DIR / "gemma4_31b_unsloth_val_scored.csv"
FULL_VAL_SUMMARY_PATH = OUTPUT_DIR / "gemma4_31b_unsloth_val_summary.csv"

if RUN_FULL_VALIDATION:
    full_val_pred_df = run_generation_with_checkpoints(
        val_df,
        FULL_VAL_PATH,
        checkpoint_every=25,
        overwrite=False,
    )
    full_val_scored_df, full_val_summary_df = score_predictions(full_val_pred_df, val_df)
    full_val_scored_df.to_csv(FULL_VAL_SCORED_PATH, index=False)
    full_val_summary_df.to_csv(FULL_VAL_SUMMARY_PATH, index=False)
    display(full_val_summary_df.round(4))
else:
    print("RUN_FULL_VALIDATION is False. Using sample_scored_df for error analysis.")
    full_val_scored_df = sample_scored_df.copy()
    full_val_summary_df = sample_summary_df.copy()

RUN_FULL_VALIDATION is False. Using sample_scored_df for error analysis.


In [20]:
def add_error_flags(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()
    out["prediction"] = out["prediction"].fillna("").astype(str)
    out["pred_words"] = out["prediction"].str.split().str.len()
    out["ref_words"] = out["output"].fillna("").astype(str).str.split().str.len()
    out["is_empty"] = out["prediction"].str.strip().eq("")
    out["maybe_too_short"] = out["pred_words"] < 0.35 * out["ref_words"].clip(lower=1)
    out["maybe_too_long"] = out["pred_words"] > 2.5 * out["ref_words"].clip(lower=1)
    out["contains_refusal"] = out["prediction"].str.contains(
        r"cannot|can't|unable|consult a doctor|medical professional|as an ai",
        case=False,
        regex=True,
    )
    return out

analysis_df = add_error_flags(full_val_scored_df)
display(analysis_df[["is_empty", "maybe_too_short", "maybe_too_long", "contains_refusal"]].mean().to_frame("rate"))
lowest_cols = ["ID", "subset", "input", "output", "prediction", "rouge1_f1", "rougeL_f1", "pred_words", "ref_words"]
display(analysis_df.sort_values("rougeL_f1").head(10)[lowest_cols])

,rate
is_empty,0.0000
maybe_too_short,0.1875
maybe_too_long,0.0375
contains_refusal,0.0500


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1,pred_words,ref_words
12,ID_VL_Amh_Eth_4DD8EB85,Amh_Eth,ገና 14 ዓመቴ ነው፣ እና ክሊኒኩ ትልቅ ሰው ማምጣት እንዳለብኝ ነገረኝ። አልችልም፣ ምን ማድረግ ይገባኛል?,ማህበራዊ ሰራተኛ ወይም ልዩ የጤና ሰራተኛ ይጠይቁ። ለአካለ መጠን ያልደረሱ ልጆች ያለ ቤተሰባቸው ሚስጥራዊ እንክብካቤ እንዲያገኙ ለመርዳት የሰለጠኑ ናቸው።,አንዳንድ ክሊኒኮች የሕግ ወይም የፖሊሲ ደንብ ሊኖራቸው ይችላል። ነገር ግን፣ በኢትዮጵያ ሕግ መሠረት፣ አብዛኞቹ የጤና ተቋማት በምስጢር የሚጠበቅ የሥነ-ተዋልዶ ጤና አገልግሎትን ለታዳጊዎች ያለ አጃቢ መስጠት አለባቸው። ሌላ ክሊኒክ ወይም የጤና ጣቢያ ይፈልጉ፣ ወይም ከጤና ባለሙያው ጋር በግልጽ ይነጋገሩ።<turn|>,0.00000,0.00000,38,19
13,ID_VL_Amh_Eth_670ACC5C,Amh_Eth,ለተጨማሪ ደህንነት ሁለት ኮንዶም በአንድ ጊዜ (አንዱ በሌላው ላይ) መጠቀም ደህንነቱ የተጠበቀ ነው?,አይሁለት ኮንዶም መጠቀም አንዱን ከመጠቀም ያነሰ ደህንነቱ የተጠበቀ ነው፣ ምክንያቱም በንብርብሮች መካከል ያለው ግጭት ሁለቱንም በቀላሉ እንዲቀደዱ ሊያደርግ ይችላል። ሁል ጊዜ ነጠላ ኮንዶም በትክክል ይጠቀሙ።,አይደለም። ሁለት ኮንዶም በአንድ ጊዜ መጠቀም ደህንነቱ የተጠበቀ አይደለም። ሁለቱ ኮንዶሞች እርስ በርሳቸው ሲፋተኩ ለመቀደድ እና ለመበጠስ ይዳርጋሉ፣ ይህም የመከላከያ አቅማቸውን ይቀንሰዋል። አንድ ኮንዶም ብቻ በትክክል መጠቀም በቂ ነው።<turn|>,0.00000,0.00000,29,25
14,ID_VL_Amh_Eth_6A5D932B,Amh_Eth,ትሪኮሞኒያሲስ የሚያመጣው ምን ዓይነት አካል ነው?,ነጠላ-ሕዋስ ፕሮቶዞአን ጥገኛ ተውሳክ (Trichomonas vaginalis)፡፡,ትሪኮሞኒያሲስ የሚከሰተው ትሪኮሞናስ ቫጂናሊስ በተባለ ፕሮቶዞዋ (አካላዊ ተህዋሲያን) ነው።<turn|>,0.00000,0.00000,9,6
15,ID_VL_Amh_Eth_7A8D03AA,Amh_Eth,ሄርፒስ እንዴት ይተላለፋል?,ሄርፒስ ምንም አይነት ምልክት ባይኖርዉም እንኳ የሚተላለፈው ቆዳ ለቆዳ በመነካካት፣ በመሳሳም ወይም በጾታዊ ግንኙነት ነው።,ሄርፒስ በቆዳ ንክኪ፣ በሳም፣ በግብረ-ሥጋ ግንኙነት እና ከበሽበሽ ቁስሎች ጋር በቀጥታ በመገናኘት ይተላለፋል።<turn|>,0.00000,0.00000,13,15
11,ID_VL_Amh_Eth_409D3602,Amh_Eth,በፕሮቶዞአን ጥገኛ ተውሳክ የሚከሰተው የትኛው ዓይነት የአባላዘር በሽታ ነው?,ትሪኮሞኒየስ (Trichomonas vaginalis)፡፡,ትሪኮሞኒያሲስ<turn|>,0.00000,0.00000,1,3
18,ID_VL_Amh_Eth_AB8DDEB4,Amh_Eth,በጤና ጣቢያው ውስጥ ለነፍሰ ጡር ታዳጊ ልጃገረዶች ምን ዓይነት አገልግሎቶች ነፃ ናቸው?,የቅድመ ወሊድ እንክብካቤ ምርመራዎች፣ የቴታነስ ክትባቶች፣ የብረት/ፎሊክ አሲድ ተጨማሪዎች እና ልጅ መውለድ ብዙ ጊዜ ነፃ ወይም ዝቅተኛ ዋጋ አላቸው። ለበለጠ መረጃ የአካባቢዎን ክሊኒክ ያነጋግሩ።,በጤና ጣቢያው ውስጥ የሚሰጡት አብዛኞቹ አገልግሎቶች ነፃ ናቸው። ይህም የቅድመ ወሊድ ምርመራን፣ የላብራቶሪ ምርመራዎችን፣ የክትባት አገልግሎቶችን እና የጤና ምክርን ያካትታል።<turn|>,0.00000,0.00000,20,24
17,ID_VL_Amh_Eth_A5CB63A2,Amh_Eth,በሰውነቴ እና በመልክዬ ኩራት ይሰማኛል?,አይከንቱ ወይም ጉራ መሆን ስህተት ነው፣ ነገር ግን የሰውነትዎን ጥንካሬ እና ጤንነት ማድነቅ ለእግዚአብሔር ፍጥረት አክብሮት ያሳያል።,አዎ፣ በራስዎ መተማመን እና ኩራት ሊኖርዎት ይገባል። እያንዳንዱ ሰው ልዩ እና ውብ ነው። ራስዎን ከሌሎች ጋር አያወዳድሩ።<turn|>,0.00000,0.00000,17,17
16,ID_VL_Amh_Eth_9DAC6E83,Amh_Eth,ትሪኮሞኒያሲስ ለኤችአይቪ ተጋላጭነትን ይጨምራል?,አዎ። ትሪኮሞኒያሲስ ካለብዎት እና ለኤችአይቪ ከተጋለጡ በቫይረሱ የመያዝ ዕድልዎን ይበልጥ ከፍተኛ ያደርገዋል።,አዎ። ትሪኮሞኒያሲስ የሴት ብልት አካልን እና የሽንት ቧንቧን ያቆስላል፣ ይህም ኤችአይቪ ወደ ሰውነት በቀላሉ እንዲገባ ያደርጋል።<turn|>,0.00000,0.00000,16,12
19,ID_VL_Amh_Eth_B04D3376,Amh_Eth,አንድ ሕፃን ከተወለደ በኋላ ወዲያውኑ የዓይን ሕመም ቢይዝ ይህ ሕመም ከእናቱ የተላለፈ ሊሆን ይችላል?,አዎ፣ ሊሆን ይችላል። በሕፃናት ላይ የሚከሰት የአይን ኢንፌክሽኖች (neonatal conjunctivitis)በዋነኝነት ምክንያቱ በወሊድ ጊዜ በሚተላለፍ ክላሚዲያ ወይም ጨብጥ ነው። ዓይነ ስውርነትን ለመከላከል ወዲያውኑ መታከም አለበት።,አዎ፣ እንደ ክላሚዲያ እና ጨብጥ ያሉ ኢንፌክሽኖች በወሊድ ወቅት ከእናቷ ወደ ልጇ ሊተላለፉ ይችላሉ።<turn|>,0.00000,0.00000,14,24
60,ID_VL_Lug_Uga_24D38751,Lug_Uga,Omuntu alina kweyisa atya oluvannyuma lw’okwegatta n’omuntu alina Akawuka akaleeta Siriimu?,"Bw’oba weenyigidde mu kwegatta n’omuntu gw’oteebereza oba gw’omanyi nti alina Akawuka akaleeta Siriimu, kyetaagisa okubaako by'okola amangu ddala okukuuma obulamu bwo n’embeera yo eya bulijjo nga nnungi. Bino by’osaanidde okulowoozaako ...","Okugenda mu ddagala eriweweeza ku Kawuka akaleeta Siriimu, okukozesa obupiira, n’okwekebeza.<turn|>",0.08284,0.08284,11,139


## 8. Test Prediction File

Run test inference only after the smoke and validation sample look better than the zero-shot baseline. The same answer is duplicated across all target columns.

In [21]:
RUN_TEST_INFERENCE = True

TEST_PRED_PATH = OUTPUT_DIR / "gemma4_31b_unsloth_test_predictions.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submission_gemma4_31b_unsloth_lora.csv"

if RUN_TEST_INFERENCE:
    test_pred_df = run_generation_with_checkpoints(
        test_df,
        TEST_PRED_PATH,
        checkpoint_every=25,
        overwrite=False,
    )
else:
    print("RUN_TEST_INFERENCE is False.")
    if TEST_PRED_PATH.exists():
        test_pred_df = pd.read_csv(TEST_PRED_PATH)
        print("Loaded existing test predictions:", TEST_PRED_PATH)
    else:
        test_pred_df = None
        print("No existing test predictions found.")

  0%|          | 0/2618 [00:00<?, ?it/s]

Saved 2618 predictions to /content/multilingual-health-question-answering/outputs/gemma4_31b_unsloth_test_predictions.csv
Elapsed: 304.4 minutes


In [22]:
def create_prediction_file(test_predictions: pd.DataFrame, sample_submission: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    if test_predictions is None or test_predictions.empty:
        raise ValueError("test_predictions is empty. Run test inference first.")

    missing = {"ID", "prediction"} - set(test_predictions.columns)
    if missing:
        raise ValueError(f"test_predictions is missing columns: {sorted(missing)}")

    pred_map = dict(zip(test_predictions["ID"], test_predictions["prediction"].fillna("")))
    missing_ids = [row_id for row_id in sample_submission["ID"] if row_id not in pred_map]
    if missing_ids:
        raise ValueError(f"Missing predictions for {len(missing_ids)} IDs. First few: {missing_ids[:5]}")

    answers = [clean_generation(pred_map[row_id]) for row_id in sample_submission["ID"]]
    if any(not ans for ans in answers):
        raise ValueError("Found empty predictions. Fix before creating the prediction file.")

    prediction_file = pd.DataFrame({
        "ID": sample_submission["ID"],
        "TargetRLF1": answers,
        "TargetR1F1": answers,
        "TargetLLM": answers,
    })

    expected_cols = ["ID", "TargetRLF1", "TargetR1F1", "TargetLLM"]
    assert prediction_file.columns.tolist() == expected_cols
    assert len(prediction_file) == len(sample_submission)
    assert prediction_file["TargetRLF1"].equals(prediction_file["TargetR1F1"])
    assert prediction_file["TargetRLF1"].equals(prediction_file["TargetLLM"])
    assert not prediction_file.isna().any().any()

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    prediction_file.to_csv(output_path, index=False)
    print("Saved prediction file:", output_path)
    return prediction_file

if test_pred_df is not None:
    prediction_file_df = create_prediction_file(test_pred_df, sample_sub_df, SUBMISSION_PATH)
    display(prediction_file_df.head())
else:
    print("Prediction file not created because test_pred_df is None.")

Saved prediction file: /content/multilingual-health-question-answering/outputs/submission_gemma4_31b_unsloth_lora.csv


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma no bi ne: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma a ɛfa GBV ho a wɔde bɛma wɔ sukuu ahorow...","Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma no bi ne: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma a ɛfa GBV ho a wɔde bɛma wɔ sukuu ahorow...","Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma no bi ne: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma a ɛfa GBV ho a wɔde bɛma wɔ sukuu ahorow..."
1,ID_TS_Aka_Gha_1C80317F,"Mmabun wɔ hokwan sɛ wɔbɛnya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu so no mu, na ɛho hia sɛ wɔbɛhwehwɛ sɛ wɔbɛma wɔn ahwɛ so yie na wɔde wɔn ho bɛhyɛ mu. Ebia ɛho hia sɛ wɔbɛhwehwɛ mmoa afi mpanyimfo, afotufo, ...","Mmabun wɔ hokwan sɛ wɔbɛnya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu so no mu, na ɛho hia sɛ wɔbɛhwehwɛ sɛ wɔbɛma wɔn ahwɛ so yie na wɔde wɔn ho bɛhyɛ mu. Ebia ɛho hia sɛ wɔbɛhwehwɛ mmoa afi mpanyimfo, afotufo, ...","Mmabun wɔ hokwan sɛ wɔbɛnya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu so no mu, na ɛho hia sɛ wɔbɛhwehwɛ sɛ wɔbɛma wɔn ahwɛ so yie na wɔde wɔn ho bɛhyɛ mu. Ebia ɛho hia sɛ wɔbɛhwehwɛ mmoa afi mpanyimfo, afotufo, ..."
2,ID_TS_Aka_Gha_06671AD1,Mmabun betumi ahwɛ nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ho denam: Nsusuanso a ɛfa nea ɛfata sɛ wɔyɛ ho nhwehwɛmu a wɔbɛhwehwɛ mu de ahwɛ sɛ wɔbɛboa anaa wɔbɛgyina ho kɛkɛ. Nneɛma a wɔbɛhwehwɛ so no bi ne: Ɔbar...,Mmabun betumi ahwɛ nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ho denam: Nsusuanso a ɛfa nea ɛfata sɛ wɔyɛ ho nhwehwɛmu a wɔbɛhwehwɛ mu de ahwɛ sɛ wɔbɛboa anaa wɔbɛgyina ho kɛkɛ. Nneɛma a wɔbɛhwehwɛ so no bi ne: Ɔbar...,Mmabun betumi ahwɛ nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ho denam: Nsusuanso a ɛfa nea ɛfata sɛ wɔyɛ ho nhwehwɛmu a wɔbɛhwehwɛ mu de ahwɛ sɛ wɔbɛboa anaa wɔbɛgyina ho kɛkɛ. Nneɛma a wɔbɛhwehwɛ so no bi ne: Ɔbar...
3,ID_TS_Aka_Gha_BDD640FB,"Amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu no betumi aka mmaabun nkitahodi ne akannifo nnwuma wɔ ASRH mu denam: Amammerɛ mu gyidi ahorow a ɛka obiara nkitahodi ne nea wogye tom ho. Asetena mu s...","Amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu no betumi aka mmaabun nkitahodi ne akannifo nnwuma wɔ ASRH mu denam: Amammerɛ mu gyidi ahorow a ɛka obiara nkitahodi ne nea wogye tom ho. Asetena mu s...","Amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu no betumi aka mmaabun nkitahodi ne akannifo nnwuma wɔ ASRH mu denam: Amammerɛ mu gyidi ahorow a ɛka obiara nkitahodi ne nea wogye tom ho. Asetena mu s..."
4,ID_TS_Aka_Gha_46685257,"Mmara nsesaeɛ ho dwumadi no hia ma mmabun nyin wɔ biribiara mu efisɛ ɛma wɔn tumi sɛ wɔbɛsakra mmara ne nhyehyɛe ahorow a ɛfa wɔn awo akwahosan ho. Eyi betumi de nsakrae pa aba wɔn awo akwahosan ho nhyehyɛe ahorow mu, a nea ɛka ho ne mm...","Mmara nsesaeɛ ho dwumadi no hia ma mmabun nyin wɔ biribiara mu efisɛ ɛma wɔn tumi sɛ wɔbɛsakra mmara ne nhyehyɛe ahorow a ɛfa wɔn awo akwahosan ho. Eyi betumi de nsakrae pa aba wɔn awo akwahosan ho nhyehyɛe ahorow mu, a nea ɛka ho ne mm...","Mmara nsesaeɛ ho dwumadi no hia ma mmabun nyin wɔ biribiara mu efisɛ ɛma wɔn tumi sɛ wɔbɛsakra mmara ne nhyehyɛe ahorow a ɛfa wɔn awo akwahosan ho. Eyi betumi de nsakrae pa aba wɔn awo akwahosan ho nhyehyɛe ahorow mu, a nea ɛka ho ne mm..."


## Run Notes

Record these values after each run:

- Base model: `unsloth/gemma-4-31B-it` or fallback model if used
- Train rows: value of `TRAIN_SAMPLE_SIZE`
- LoRA: `r`, `alpha`, dropout
- Training steps or epochs
- Validation sample size
- ROUGE-1 F1 and ROUGE-L F1
- Public leaderboard metrics after upload
- Runtime and peak GPU memory

Compare against the current zero-shot Gemma 4 E4B baseline before committing to a longer training run.